In [47]:
import numpy as np
import pandas as pd
import json
import random
from itertools import combinations
from itertools import combinations_with_replacement
from pathlib import Path
import sqlite3

In [48]:
## 1. CREATE DATABASE
# Database location
db_path = Path("../data/thanksgiving_point.db")

# Create/connect to SQLite database
conn = sqlite3.connect(db_path)
print(f"Database created at: {db_path.resolve()}")


Database created at: /Users/raul/Desktop/coding/DevProjects/YearlyImpact Research/data/thanksgiving_point.db


In [49]:
## 2. CREATE TABLES
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS households;
DROP TABLE IF EXISTS children;
DROP TABLE IF EXISTS visits;
DROP TABLE IF EXISTS survey_responses;

CREATE TABLE households (
    household_id TEXT PRIMARY KEY,
    family_size INTEGER NOT NULL,
    adults INTEGER NOT NULL,
    children_count INTEGER NOT NULL,
    average_child_age REAL
);

CREATE TABLE children (
    child_id TEXT PRIMARY KEY,
    household_id TEXT NOT NULL,
    age INTEGER NOT NULL,
    FOREIGN KEY (household_id) REFERENCES households(household_id)
);

CREATE TABLE visits (
    visit_id TEXT PRIMARY KEY,
    household_id TEXT NOT NULL,
    visit_date TEXT NOT NULL,
    visit_number INTEGER NOT NULL,
    exhibit TEXT,
    crowd_density REAL,
    number_of_programs INTEGER,
    spending REAL,
    FOREIGN KEY (household_id) REFERENCES households(household_id)
);

CREATE TABLE survey_responses (
    survey_id TEXT PRIMARY KEY,
    visit_id TEXT NOT NULL,
    household_id TEXT NOT NULL,
    satisfaction REAL,
    learning_score REAL,
    curiosity_score REAL,
    confidence_score REAL,
    social_interaction_score REAL,
    return_intent INTEGER,
    recommend_score INTEGER,
    comments TEXT,
    FOREIGN KEY (visit_id) REFERENCES visits(visit_id)
);
""")

conn.commit()

print("Tables created.")

Tables created.


In [50]:
## 3. HOUSEHOLDS DATA
np.random.seed(42)
n_households, households = 100000, []

for i in range(n_households):
    household_id = f"H{i+1:05d}"
    family_size = np.random.choice([2, 3, 4, 5, 6],p=[0.08, 0.17, 0.30, 0.28, 0.17])

    # Maximum of 2 adults
    adults = min(2, np.random.choice([1, 2], p=[0.15, 0.85]))

    # Make sure children >= 0
    children_count = family_size - adults
    households.append([ household_id,family_size,adults,children_count])

households_df = pd.DataFrame(households,columns=["household_id","family_size","adults","children_count"])

households_df.head()


,household_id,family_size,adults,children_count
0,H00001,4,2,2
1,H00002,5,2,3
2,H00003,3,2,1
3,H00004,2,2,0
4,H00005,5,2,3


In [51]:
## 4. CHILDREN DATA
children = []

for _, household in households_df.iterrows():
    household_id = household["household_id"]
    n_children = household["children_count"]

    if n_children == 0:
        continue

    # Random target average age
    target_average = np.random.randint(3, 17)

    # Generate ages around the target
    ages = np.random.randint(
        max(1, target_average - 3),
        min(18, target_average + 4),
        size=n_children
    )

    if n_children == 1:
        age_combinations = [
            (2,),
            (3,),
            (4,),
            (5,),
            (6,)
        ]
    
        ages = age_combinations[np.random.choice(len(age_combinations),p=[0.08, 0.17, 0.30, 0.28, 0.17])]
    
    
    elif n_children == 2:
        age_combinations = [
            (4, 6),
            (3, 6),
            (4, 8),
            (5, 11),
            (1, 2)
        ]
    
        ages = age_combinations[np.random.choice(len(age_combinations),p=[0.10, 0.20, 0.30, 0.25, 0.15])]
    
    
    elif n_children == 3:
        age_combinations = [
            (4, 6, 8),
            (3, 6, 9),
            (2, 3, 7),
            (5, 9, 10),
            (2, 6, 10)
        ]
    
        ages = age_combinations[np.random.choice(len(age_combinations),p=[0.10, 0.20, 0.30, 0.25, 0.15])]
    
    
    elif n_children == 4:
        age_combinations = [
            (2, 4, 6, 8),
            (3, 6, 9, 12),
            (2, 4, 8, 10),
            (2, 6, 10, 14),
            (1, 7, 9, 11)
        ]
    
        ages = age_combinations[np.random.choice(len(age_combinations),p=[0.10, 0.20, 0.30, 0.25, 0.15])]
    
    
    elif n_children == 5:
        age_combinations = [
            (1, 2, 3, 4, 5),
            (6, 7, 8, 9, 10),
            (2, 5, 8, 11, 14),
            (11, 12, 13, 14, 15),
            (4, 6, 8, 10, 12)
        ]
    
        ages = age_combinations[ np.random.choice(len(age_combinations),p=[0.10, 0.20, 0.30, 0.25, 0.15])]

    for j, age in enumerate(ages):

        children.append([
            f"{household_id}_C{j+1}",
            household_id,
            int(age)
        ])

children_df = pd.DataFrame(
    children,
    columns=[
        "child_id",
        "household_id",
        "age"
    ]
)

average_ages = (
    children_df
    .groupby("household_id")["age"]
    .mean()
    .reset_index()
    .rename(columns={"age": "average_child_age"})
)

households_df = households_df.merge(
    average_ages,
    on="household_id",
    how="left"
)

children_df

,child_id,household_id,age
0,H00001_C1,H00001,1
1,H00001_C2,H00001,2
2,H00002_C1,H00002,4
3,H00002_C2,H00002,6
4,H00002_C3,H00002,8
...,...,...,...
244137,H99999_C3,H99999,9
244138,H100000_C1,H100000,2
244139,H100000_C2,H100000,6
244140,H100000_C3,H100000,10


In [52]:
## 4. CHILDREN DATA

visits = []

exhibits = ["Curiosity Museum","Gardens","Farm","Dinosaur Museum","Butterfly Biosphere"]

for _, household in households_df.iterrows():
    household_id = household["household_id"]
    
    # Most households visit 1–4 times
    number_of_visits = np.random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],p=[.3,.23,.15,.1,.07,.05,.035,.025,.015,.012,.009,.004])

    for visit_number in range(1, number_of_visits + 1):

        visit_id = f"{household_id}_V{visit_number}"
        visit_date = (pd.Timestamp("2026-01-01")+ pd.Timedelta(days=np.random.randint(0, 260)))
        exhibit = np.random.choice(exhibits)
        crowd_density = np.random.triangular(left=5,mode=50,right=100)
        number_of_programs = np.random.choice([0, 1, 2, 3, 4, 5],p=[0.40, 0.30, 0.15, 0.08, 0.05, 0.02])        
        spending = max(10,np.random.normal(55, 25))
        visits.append([
            visit_id,
            household_id,
            visit_date.strftime("%Y-%m-%d"),
            visit_number,
            exhibit,
            crowd_density,
            number_of_programs,
            round(spending, 2)
        ])

visits_df = pd.DataFrame(
    visits,
    columns=[
        "visit_id",
        "household_id",
        "visit_date",
        "visit_number",
        "exhibit",
        "crowd_density",
        "number_of_programs",
        "spending"
])

visits_df


,visit_id,household_id,visit_date,visit_number,exhibit,crowd_density,number_of_programs,spending
0,H00001_V1,H00001,2026-01-18,1,Gardens,44.150495,4,39.63
1,H00001_V2,H00001,2026-09-03,2,Gardens,43.584904,3,71.70
2,H00001_V3,H00001,2026-09-06,3,Butterfly Biosphere,19.008313,2,63.19
3,H00001_V4,H00001,2026-07-15,4,Gardens,71.821010,0,66.13
4,H00001_V5,H00001,2026-05-15,5,Dinosaur Museum,35.720337,1,53.73
...,...,...,...,...,...,...,...,...
308892,H99998_V2,H99998,2026-03-02,2,Farm,53.604600,0,60.42
308893,H99998_V3,H99998,2026-01-26,3,Dinosaur Museum,67.325803,0,56.08
308894,H99998_V4,H99998,2026-06-11,4,Dinosaur Museum,62.652083,0,68.80
308895,H99999_V1,H99999,2026-08-24,1,Curiosity Museum,69.851207,0,75.38


In [53]:
## 5. Comments
comments = {
    "positive": [
        "We had a wonderful time as a family.",
        "The kids loved the experience.",
        "Everything was fun and well organized.",
        "We really enjoyed our visit.",
        "The children were excited the whole time."
    ],

    "learning": [
        "The kids learned a lot from the exhibits.",
        "We enjoyed the hands-on learning activities.",
        "The educational programs were very informative.",
        "My children were curious about the exhibits.",
        "The activities helped the kids understand new concepts."
    ],

    "crowding": [
        "It was very crowded today.",
        "Some areas were difficult to move through.",
        "There were too many people in some exhibits.",
        "The crowds made it harder for the kids to participate.",
        "It would be better with fewer people."
    ],

    "program": [
        "The educational program was excellent.",
        "The kids really enjoyed the program.",
        "The program kept everyone engaged.",
        "We learned a lot from the activity.",
        "The children wanted to participate again."
    ],

    "negative": [
        "Some areas were disappointing.",
        "The experience was not as good as we expected.",
        "There were not enough activities for younger children.",
        "Some exhibits needed better maintenance.",
        "The visit felt a little disorganized."
    ],

    "return": [
        "We would definitely come back.",
        "We are looking forward to visiting again.",
        "The kids want to come back soon.",
        "We would recommend this experience to other families.",
        "We enjoyed it enough to plan another visit."
    ]
}

all_comments = []

for i in range(1500):
    category = np.random.choice(list(comments.keys()),p=[0.25, 0.20, 0.15, 0.15, 0.10, 0.15])
    all_comments.append(np.random.choice(comments[category]))

with open("../data/comments.json", "w", encoding="utf-8") as f:
    json.dump(all_comments, f, indent=4, ensure_ascii=False)

In [54]:
## 6. EACH VISIT DATA

with open("../data/comments.json", "r", encoding="utf-8") as f:
    all_comments = json.load(f)

survey_responses = []


for _, visit in visits_df.iterrows():
    household_id = visit["household_id"]
    # Satisfaction
    satisfaction = np.clip(np.random.normal(4.0, 0.7),1,5)

    # Learning
    learning = np.clip(3.2+ 0.35 * visit["number_of_programs"]+ np.random.normal(0, 0.6),1,5)

    # Curiosity
    curiosity = np.clip(3.4+ 0.25 * learning+ np.random.normal(0, 0.6),1,5)

    # Confidence
    confidence = np.clip(3.3+ 0.20 * learning+ np.random.normal(0, 0.6),1,5)

    # Social interaction
    social = np.clip(3.5+ np.random.normal(0, 0.6),1,5)

    # Return intent
    return_score = np.clip(round(satisfaction+ 0.3 * learning- 0.03 * visit["crowd_density"]+ np.random.normal(0, 0.5)),1,5)

    recommend = np.clip(round(satisfaction+ np.random.normal(0, 0.5)),1,5)
    comment = np.random.choice(all_comments)

    survey_responses.append([
        f"{visit['visit_id']}_S1",
        visit["visit_id"],
        household_id,
        round(satisfaction, 2),
        round(learning, 2),
        round(curiosity, 2),
        round(confidence, 2),
        round(social, 2),
        return_score,
        recommend,
        comment
    ])

survey_df = pd.DataFrame(
    survey_responses,
    columns=[
        "survey_id",
        "visit_id",
        "household_id",
        "satisfaction",
        "learning_score",
        "curiosity_score",
        "confidence_score",
        "social_interaction_score",
        "return_intent",
        "recommend_score",
        "comments"
    ]
)

survey_df


,survey_id,visit_id,household_id,satisfaction,learning_score,curiosity_score,confidence_score,social_interaction_score,return_intent,recommend_score,comments
0,H00001_V1_S1,H00001_V1,H00001,3.29,4.92,4.50,5.00,3.56,3,4,There were too many people in some exhibits.
1,H00001_V2_S1,H00001_V2,H00001,3.03,4.56,5.00,3.76,3.33,3,2,We learned a lot from the activity.
2,H00001_V3_S1,H00001_V3,H00001,3.96,3.45,4.23,4.48,4.40,5,3,The kids loved the experience.
3,H00001_V4_S1,H00001_V4,H00001,3.66,3.02,3.93,2.93,3.89,3,4,It would be better with fewer people.
4,H00001_V5_S1,H00001_V5,H00001,3.15,2.88,4.33,4.59,3.53,3,3,We enjoyed the hands-on learning activities.
...,...,...,...,...,...,...,...,...,...,...,...
308892,H99998_V2_S1,H99998_V2,H99998,5.00,3.49,4.67,4.38,3.04,4,5,We are looking forward to visiting again.
308893,H99998_V3_S1,H99998_V3,H99998,4.17,2.78,4.88,4.10,3.23,2,4,We really enjoyed our visit.
308894,H99998_V4_S1,H99998_V4,H99998,3.50,3.76,4.02,2.83,3.54,3,3,We are looking forward to visiting again.
308895,H99999_V1_S1,H99999_V1,H99999,4.99,2.63,3.89,4.04,4.14,4,4,The crowds made it harder for the kids to part...


In [55]:
## 7. SAVE THE DATA TO THE DATABASE
try: 
    households_df.to_sql("households",conn,if_exists="append",index=False)
    children_df.to_sql("children",conn,if_exists="append",index=False)
    visits_df.to_sql("visits",conn,if_exists="append",index=False)
    survey_df.to_sql("survey_responses",conn,if_exists="append",index=False)

    print("Data successfully inserted into database.")

finally:
    conn.close()

Data successfully inserted into database.


In [ ]:
##